# Open Notebook in Colab

[![Open in Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/childmindresearch/llm_tracker/blob/main/tutorials/text_descriptives_tutorial.ipynb)


# Text descriptives tutorial

Explore the length, vocabulary, readability, and sentence patterns of your documents before analysis. This workflow runs locally and makes **no LLM calls**. It does not need an API key.

Run the cells from top to bottom. The default uses the bundled practice CSV. You can also use your own CSV (one document per row) or a folder of `.txt` files. The default English spaCy model is appropriate for the bundled example; use a matching model for other languages.


## Setup (once)

Run the next cell to install LLM Tracker with its `descriptives` extra and download the English spaCy model. The extra includes compatible TextDescriptives and spaCy versions; the main coding tutorial does not install these tools. Practice files ship inside the package, so no repository checkout is needed. Setup may take a few minutes and makes no LLM calls.

This works in Google Colab or a local Jupyter notebook using Python 3.11–3.13. If asked to restart the session/kernel, restart it and rerun this setup cell.


In [ ]:
%pip install "llm-tracker[descriptives] @ https://github.com/childmindresearch/llm_tracker/archive/refs/heads/main.zip"

import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "spacy", "download", "en_core_web_sm"],
    check=True,
)


## 1. Choose documents

The default practice file is a CSV with one document per row and a `text` column. For your own data, set `input_source` to:

- `"path"` and update `input_data` to a CSV or a folder of `.txt` files available to this Python session.
- `"upload"` to select a CSV or one or more `.txt` files in Colab.
- `"drive"` and set `drive_input_data` to a path under `/content/drive/MyDrive/`.

Colab cannot access files on your computer by local path. For persistent output in Colab, set `output_root` to a folder under `/content/drive/MyDrive/`.


In [2]:
from pathlib import Path
import tempfile
from llm_tracker.datasets import copy_demo_data

sample_dir = copy_demo_data("tutorial_data")
input_source = "demo"  # "demo", "path", "upload", or "drive".
input_data = sample_dir / "documents.csv"  # CSV or a folder of .txt files for "path".
drive_input_data = Path("/content/drive/MyDrive/study/documents.csv")
text_column = "text"
output_root = Path("research_results")
spacy_model = "en_core_web_sm"


In [3]:
if input_source == "demo":
    source_path = sample_dir / "documents.csv"
elif input_source == "path":
    source_path = Path(input_data).expanduser().resolve()
elif input_source == "drive":
    from google.colab import drive

    drive.mount("/content/drive")
    source_path = Path(drive_input_data).expanduser().resolve()
    if not source_path.is_relative_to(Path("/content/drive/MyDrive")):
        raise ValueError("Set drive_input_data to a path under /content/drive/MyDrive/.")
elif input_source == "upload":
    from google.colab import files

    uploaded = files.upload()
    if not uploaded:
        raise ValueError("No documents selected.")
    text_files = all(Path(name).suffix.lower() == ".txt" for name in uploaded)
    if not text_files and len(uploaded) != 1:
        raise ValueError("Upload one CSV file or only TXT files.")
    upload_dir = Path(tempfile.mkdtemp(prefix="llm_tracker_text_descriptives_"))
    for name, contents in uploaded.items():
        (upload_dir / Path(name).name).write_bytes(contents)
    source_path = upload_dir if text_files else upload_dir / Path(next(iter(uploaded))).name
else:
    raise ValueError('Choose input_source = "demo", "path", "upload", or "drive".')

if not source_path.exists():
    raise FileNotFoundError(f"Documents not found: {source_path}")
if source_path.is_dir():
    from llm_tracker.file_handlers import get_document_files, load_document

    document_paths = get_document_files(source_path)
    if not document_paths:
        raise ValueError(f"No supported TXT documents found in {source_path}.")
    documents = [load_document(path)[0] for path in document_paths]
else:
    import pandas as pd

    input_frame = pd.read_csv(source_path)
    if text_column not in input_frame:
        raise ValueError(f"Text column '{text_column}' was not found in {source_path}.")
    if input_frame.empty:
        raise ValueError(f"No documents found in {source_path}.")
    if input_frame[text_column].isna().any():
        raise ValueError(f"The '{text_column}' column contains missing document text.")
    documents = input_frame[text_column].astype(str).tolist()

print(f"Loaded {len(documents)} documents from {source_path}")


Loaded 4 documents from /Users/freymon.perez/Projects/GitHub/llm_tracker/sample_data/tutorial_practice/documents.csv


## 2. Summarize the corpus

The summary table reports corpus-level statistics. A timestamped CSV with measurements for each document is also saved under `output_root`.


In [4]:
from IPython.display import display
from llm_tracker.corpus_summary import summarize_corpus

output_root = Path(output_root).expanduser().resolve()
text_summary, text_by_document = summarize_corpus(
    documents,
    spacy_model=spacy_model,
    include_readability=True,
    include_sentence_complexity=True,
    include_coherence_sbert=False,
    output_dir=output_root,
    input_path=str(source_path),
)
display(text_summary)


Extracting TextDescriptives metrics (descriptive_stats, pos_proportions, readability, dependency_distance)…
Computing parse-tree depths…


,Metric,Value
0,N (documents),4
1,Sentences per document,1.25 (0.50) [1.00–2.00]
2,Sentence length (words),13.38 (2.14) [11.00–16.00]
3,Tokens per document,16.50 (6.03) [11.00–25.00]
4,Unique tokens per document,14.75 (3.86) [11.00–20.00]
5,Type-token ratio (TTR),0.92 (0.08) [0.80–1.00]
6,Characters per document,67.75 (20.61) [51.00–97.00]
7,1st-person pronoun proportion,22.15% (3.04%) [18.18%–25.00%]
8,Proportion NOUN,18.21% (4.91%) [13.33%–25.00%]
9,Proportion VERB,22.44% (4.43%) [16.67%–26.67%]
